# Word–score association rules on the 09f sample

Find words and two-word phrases associated with the top-decile `p_support` and `p_skeptic` model scores. Discover rules in training comments and recalculate them on held-out posts. These are patterns in model scores, not verified human stances.


In [1]:
%pip install pandas pyarrow scikit-learn mlxtend

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import GroupShuffleSplit
from mlxtend.frequent_patterns import fpgrowth, association_rules

DATA_PATH = Path(
    r"E:\Dataminingproject\Data\09f_interpretable_assignments_with_two_scores.parquet"
)
OUTPUT_DIR = Path(r"E:\Dataminingproject\ARM_results\after_09f")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMMUNITIES = [
    "climatechange", "climate", "energy",
    "Futurology", "conspiracy", "changemyview",
]
START = pd.Timestamp("2023-11-19")
END = pd.Timestamp("2026-09-01")  # exclusive
SEED = 42

In [3]:
# Use the same 28,230 comments selected and clustered in 09f.
columns = [
    "comment_id", "post_id", "subreddit", "created_time",
    "clean_text", "p_support", "p_skeptic",
    "primary_cluster", "topic_name_en",
]
df = pd.read_parquet(DATA_PATH, columns=columns)
assert len(df) == 28_230 and df["comment_id"].is_unique
assert set(df["subreddit"]) == set(COMMUNITIES)
assert df.groupby("subreddit").size().eq(4_705).all()
assert df["created_time"].between(START, END, inclusive="left").all()
assert df["clean_text"].notna().all() and df["clean_text"].str.strip().ne("").all()
assert df[["p_support", "p_skeptic"]].notna().all().all()
assert df[["p_support", "p_skeptic"]].ge(0).all().all()
assert df[["p_support", "p_skeptic"]].le(1).all().all()
print("09f sample size:", len(df))
display(df["subreddit"].value_counts())
display(df[["clean_text", "p_support", "p_skeptic", "topic_name_en"]].head())


09f sample size: 28230


subreddit
Futurology       4705
changemyview     4705
conspiracy       4705
climatechange    4705
climate          4705
energy           4705
Name: count, dtype: int64

,clean_text,p_support,p_skeptic,topic_name_en
0,The AI would benefit from knowing what people ...,0.001255,0.000271,Fossil fuels and transition political economy
1,I think bizjets are really awesome and I will ...,0.023509,0.017078,"Emission responsibility, consumption, and glob..."
2,You think climate change fears didn't exist in...,0.040655,0.003241,"Future climate risk, adaptation, and human con..."
3,You seem to be under the impression that the w...,0.558365,0.000869,"Future climate risk, adaptation, and human con..."
4,The world has to survive to look back on anyth...,0.946300,0.015102,"Future climate risk, adaptation, and human con..."


In [4]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_index, test_index = next(splitter.split(df, groups=df["post_id"]))
train = df.iloc[train_index].reset_index(drop=True)
test = df.iloc[test_index].reset_index(drop=True)
assert set(train["post_id"]).isdisjoint(set(test["post_id"]))

# Define each top-decile cutoff on training comments only; these are model-score groups.
SCORE_QUANTILE = 0.90
SCORE_TARGETS = {
    "support_top10": "p_support",
    "skeptic_top10": "p_skeptic",
}
thresholds = {
    target: train[column].quantile(SCORE_QUANTILE)
    for target, column in SCORE_TARGETS.items()
}
for target, column in SCORE_TARGETS.items():
    train[target] = train[column].ge(thresholds[target])
    test[target] = test[column].ge(thresholds[target])
    print(
        f"{target}: training cutoff={thresholds[target]:.4f}; "
        f"training={train[target].mean():.1%}, held-out={test[target].mean():.1%}"
    )

print("Rules discovered on:", len(train))
print("Rules checked on:", len(test))
print("Comments high on both scores:", int(train[list(SCORE_TARGETS)].all(axis=1).sum()))

support_top10: training cutoff=0.9864; training=10.0%, held-out=10.0%
skeptic_top10: training cutoff=0.8070; training=10.0%, held-out=11.7%
Rules discovered on: 22501
Rules checked on: 5729
Comments high on both scores: 38


In [5]:
vectorizer = CountVectorizer(
    stop_words="english", ngram_range=(1, 2),
    min_df=0.01, max_df=0.5, max_features=200, binary=True,
)
train_words = vectorizer.fit_transform(train["clean_text"])
test_words = vectorizer.transform(test["clean_text"])
word_columns = ["word:" + word for word in vectorizer.get_feature_names_out()]

train_basket = pd.DataFrame(train_words.toarray().astype(bool), columns=word_columns)
test_basket = pd.DataFrame(test_words.toarray().astype(bool), columns=word_columns)
for target in SCORE_TARGETS:
    train_basket["score:" + target] = train[target].to_numpy()
    test_basket["score:" + target] = test[target].to_numpy()

print("Extracted words and phrases:")
print(vectorizer.get_feature_names_out())
display(train_basket.head())

Extracted words and phrases:
['10' '100' '20' 'able' 'actually' 'ago' 'agree' 'air' 'aren' 'atmosphere'
 'average' 'away' 'bad' 'based' 'batteries' 'battery' 'believe' 'best'
 'better' 'big' 'build' 'building' 'car' 'carbon' 'care' 'cars' 'cause'
 'change' 'changes' 'cheaper' 'china' 'clean' 'climate' 'climate change'
 'co2' 'coal' 'come' 'companies' 'cost' 'costs' 'countries' 'country'
 'current' 'data' 'day' 'decades' 'demand' 'did' 'didn' 'different' 'does'
 'doesn' 'doing' 'don' 'earth' 'electric' 'electricity' 'emissions' 'end'
 'energy' 'entire' 'environment' 'environmental' 'ev' 'evs' 'example'
 'expensive' 'fact' 'far' 'food' 'fossil' 'fossil fuel' 'fossil fuels'
 'fuel' 'fuels' 'future' 'gas' 'generation' 'getting' 'global'
 'global warming' 'going' 'good' 'government' 'great' 'green' 'grid'
 'happen' 'having' 'heat' 'help' 'high' 'higher' 'huge' 'human' 'humans'
 'ice' 'impact' 'increase' 'industry' 'infrastructure' 'instead' 'isn'
 'issue' 'just' 'know' 'land' 'large' 'let' 

,word:10,word:100,word:20,word:able,word:actually,word:ago,word:agree,word:air,word:aren,word:atmosphere,...,word:won,word:work,word:world,word:worse,word:wrong,word:year,word:years,word:yes,score:support_top10,score:skeptic_top10
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
3,False,False,False,True,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True


In [6]:
frequent_items = fpgrowth(
    train_basket, min_support=0.005, use_colnames=True, max_len=2,
)
if not frequent_items["itemsets"].map(len).ge(2).any():
    raise ValueError("No frequent two-item sets; check the vocabulary and sample.")

rules = association_rules(
    frequent_items, metric="confidence", min_threshold=0.20,
)
# Keep one word or phrase leading to a high model score in one dimension.
wanted = (
    rules["antecedents"].apply(
        lambda items: len(items) == 1
        and all(item.startswith("word:") for item in items)
    )
    & rules["consequents"].apply(
        lambda items: len(items) == 1
        and next(iter(items)) in {"score:" + target for target in SCORE_TARGETS}
    )
    & rules["lift"].ge(1.5)
)
rules = rules.loc[wanted].copy()
rules["word"] = rules["antecedents"].apply(
    lambda items: next(iter(items)).removeprefix("word:")
)
rules["target"] = rules["consequents"].apply(
    lambda items: next(iter(items)).removeprefix("score:")
)
rules = rules.sort_values(["lift", "support"], ascending=False).reset_index(drop=True)
display(rules[["word", "target", "support", "confidence", "lift"]].head(20))

,word,target,support,confidence,lift
0,reduce,support_top10,0.008311,0.274596,2.744864
1,science,skeptic_top10,0.009822,0.262470,2.623654
2,global warming,skeptic_top10,0.011066,0.255910,2.558072
3,renewables,support_top10,0.012044,0.245471,2.453729
4,believe,skeptic_top10,0.010800,0.238002,2.379068
5,warming,skeptic_top10,0.016266,0.232086,2.319934
6,help,support_top10,0.006977,0.229532,2.294404
7,fossil fuels,support_top10,0.013822,0.228845,2.287532
8,clean,support_top10,0.006977,0.222695,2.226060
9,green,support_top10,0.008755,0.222599,2.225099


In [7]:
checks = []
for _, rule in rules.iterrows():
    word_present = test_basket["word:" + rule["word"]]
    target_present = test_basket["score:" + rule["target"]]
    word_count = int(word_present.sum())
    joint_count = int((word_present & target_present).sum())
    baseline = float(target_present.mean())
    confidence = joint_count / word_count if word_count else np.nan
    lift = confidence / baseline if baseline > 0 else np.nan
    checks.append({
        "word": rule["word"], "target": rule["target"],
        "train_support": rule["support"],
        "train_confidence": rule["confidence"],
        "train_lift": rule["lift"],
        "test_word_count": word_count,
        "test_joint_count": joint_count,
        "test_confidence": confidence,
        "test_lift": lift,
    })

checked = pd.DataFrame(checks, columns=[
    "word", "target", "train_support", "train_confidence", "train_lift",
    "test_word_count", "test_joint_count", "test_confidence", "test_lift",
])
display(checked.head(20))

,word,target,train_support,train_confidence,train_lift,test_word_count,test_joint_count,test_confidence,test_lift
0,reduce,support_top10,0.008311,0.274596,2.744864,205,53,0.258537,2.593969
1,science,skeptic_top10,0.009822,0.262470,2.623654,244,66,0.270492,2.306023
2,global warming,skeptic_top10,0.011066,0.255910,2.558072,265,75,0.283019,2.412820
3,renewables,support_top10,0.012044,0.245471,2.453729,254,59,0.232283,2.330564
4,believe,skeptic_top10,0.010800,0.238002,2.379068,280,72,0.257143,2.192219
5,warming,skeptic_top10,0.016266,0.232086,2.319934,450,99,0.220000,1.875565
6,help,support_top10,0.006977,0.229532,2.294404,187,37,0.197861,1.985193
7,fossil fuels,support_top10,0.013822,0.228845,2.287532,364,84,0.230769,2.315371
8,clean,support_top10,0.006977,0.222695,2.226060,164,35,0.213415,2.141248
9,green,support_top10,0.008755,0.222599,2.225099,252,55,0.218254,2.189802


In [8]:
rule_output = OUTPUT_DIR / "word_score_rules_two_scores.csv"
checked.to_csv(rule_output, index=False, encoding="utf-8-sig")
print("Rule table saved to:", rule_output)


Rule table saved to: E:\Dataminingproject\ARM_results\after_09f\word_score_rules_two_scores.csv


In [9]:
# Select one rule per score group with enough held-out comments for text review.
example_rules = (
    checked.loc[checked["test_word_count"].ge(20)]
    .sort_values(["test_lift", "test_word_count"], ascending=False)
    .drop_duplicates("target")
)
display(example_rules[[
    "word", "target", "train_lift", "test_word_count", "test_lift",
]])
if len(example_rules) < len(SCORE_TARGETS):
    print("No eligible rule for one score dimension; no term is forced into the review.")

,word,target,train_lift,test_word_count,test_lift
0,reduce,support_top10,2.744864,205,2.593969
2,global warming,skeptic_top10,2.558072,265,2.412820


In [10]:
selected = []
for _, rule in example_rules.iterrows():
    word = rule["word"]
    target = rule["target"]
    has_word = test_basket["word:" + word].to_numpy()
    matching = test.loc[
        has_word,
        ["comment_id", "subreddit", "clean_text", "p_support", "p_skeptic", target],
    ].copy()
    matching["word"] = word
    matching["target"] = target

    # Sample up to 10 target-high and 10 not-target-high comments per rule for close reading.
    for category, group in [
        ("target_high", matching[matching[target]]),
        ("not_target_high", matching[~matching[target]]),
    ]:
        if group.empty:
            continue
        picked = group.sample(n=min(10, len(group)), random_state=SEED).copy()
        picked["sample_category"] = category
        selected.append(picked.drop(columns=[target]))

review_columns = [
    "word", "target", "sample_category", "subreddit",
    "p_support", "p_skeptic", "clean_text",
    "human_label", "reason_or_note", "comment_id",
]
if selected:
    review = pd.concat(selected, ignore_index=True)
    review["human_label"] = ""
    review["reason_or_note"] = ""
    review = review[review_columns]
    output = OUTPUT_DIR / "manual_review_two_scores.csv"
    review.to_csv(output, index=False, encoding="utf-8-sig")
    print("Sampled comments:", len(review))
    print("Saved to:", output)
    display(review[["word", "target", "sample_category", "subreddit",
                    "p_support", "p_skeptic", "clean_text"]])
else:
    print("No eligible rules to sample; no review file was created.")

Sampled comments: 40
Saved to: E:\Dataminingproject\ARM_results\after_09f\manual_review_two_scores.csv


,word,target,sample_category,subreddit,p_support,p_skeptic,clean_text
0,reduce,support_top10,target_high,climate,0.994870,0.000368,*More from Bloomberg News reporters Eric Rosto...
1,reduce,support_top10,target_high,climate,0.997414,0.000084,">From 2005 to 2021, US CO2 emissions were redu..."
2,reduce,support_top10,target_high,changemyview,0.999650,0.000692,Work and society are themselves a major net lo...
3,reduce,support_top10,target_high,climate,0.998016,0.001341,We dont need to address how we feel about clim...
4,reduce,support_top10,target_high,climatechange,0.997965,0.118589,These are all cool and great ideas. But the to...
5,reduce,support_top10,target_high,Futurology,0.995056,0.006364,"Using ""planting trees"" as an excuse to avoid r..."
6,reduce,support_top10,target_high,climate,0.994043,0.008678,I think that's because things are not explaine...
7,reduce,support_top10,target_high,climate,0.994840,0.059592,"Talk about all of it. It's not a choice of ""re..."
8,reduce,support_top10,target_high,conspiracy,0.997594,0.001061,Well if you zoom in on recent data from that g...
9,reduce,support_top10,target_high,climate,0.994820,0.000362,As far as I know: China proposed the 2030 carb...
